In [ ]:
import pandas as pd
import time
from nba_api.stats.endpoints import leaguedashplayerstats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
import matplotlib.pyplot as plt
import xgboost as xgb
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# -------------------------------------------------------
# NBA Raw Stats Collection
# Collects traditional + advanced per-game stats
# for every season from 2000–01 to 2024–25
# -------------------------------------------------------

def season_string(year):
    # Convert 2000 → '2000-01'
    return f"{year}-{str(year+1)[-2:]}"


def collect_single_season(year):
    # Download per-season traditional + advanced stats.
    season = season_string(year)
    print(f"Collecting {season}...")

    try:
        # Traditional
        df_trad = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            per_mode_detailed='PerGame',
            season_type_all_star='Regular Season'
        ).get_data_frames()[0]

        time.sleep(0.6)

        # Advanced
        df_adv = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            per_mode_detailed='PerGame',
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced'
        ).get_data_frames()[0]

        # Merge on PLAYER_ID
        df = pd.merge(
            df_trad,
            df_adv[['PLAYER_ID', 'OFF_RATING', 'DEF_RATING', 'NET_RATING',
                    'AST_PCT', 'REB_PCT', 'TS_PCT', 'USG_PCT', 'PIE']],
            on='PLAYER_ID',
            how='left'
        )

        df["SEASON"] = season
        df["SEASON_YEAR"] = year

        print(f"  ✓ {len(df)} players")
        return df

    except Exception as e:
        print(f"  ✗ Error collecting {season}: {e}")
        return None


# -------------------------------------------------------
# Collect all seasons and save to CSV
# -------------------------------------------------------

all_seasons = []

# Use 2000–01 through 2024–25 because it is the best balance between similarity to the modern NBA
# and amount of data
for year in range(2000, 2025):   
    df_season = collect_single_season(year)
    if df_season is not None:
        all_seasons.append(df_season)
    time.sleep(1)

# Save all years together
df_all = pd.concat(all_seasons, ignore_index=True)
df_all.to_csv("nba_raw_stats.csv", index=False)

print(f"\n✓ Saved nba_raw_stats.csv with {len(df_all)} player-seasons")


In [ ]:
# ------------------------
# Load MVP voting data (csv collected from Basketball Reference)
# ------------------------
mvp_df = pd.read_csv("NBAMVPVotingData.csv")
mvp_df["Player"] = mvp_df["Player"].str.strip()
mvp_df["Season"] = mvp_df["Season"].astype(str).str.strip()
mvp_df["Share"] = pd.to_numeric(mvp_df["Share"], errors='coerce')

# ------------------------
# Load raw NBA stats from previous block
# ------------------------
df = pd.read_csv("nba_raw_stats.csv")
df["PLAYER_NAME"] = df["PLAYER_NAME"].str.strip()
df["SEASON"] = df["SEASON"].astype(str).str.strip()
print(f"✓ Loaded raw stats ({len(df)} rows)")

# ------------------------
# Merge MVP share
# ------------------------
mvp_subset = mvp_df[["Player", "Season", "Share"]].rename(
    columns={"Player": "PLAYER_NAME", "Season": "SEASON", "Share": "MVP_SHARE"}
)

df = pd.merge(df, mvp_subset, on=["PLAYER_NAME", "SEASON"], how="left")
df["MVP_SHARE"] = df["MVP_SHARE"].fillna(0.0)
print(f"✓ Merged MVP Share. Player-seasons with votes: {(df['MVP_SHARE']>0).sum()}")

# ------------------------
# Feature engineering
# ------------------------
df["PTS_REB_AST"] = df["PTS"] + df["REB"] + df["AST"]
# I wanted to get a good approximation of a player "filling up the box score" in addition to these
# stats individually

z_stats = ['GP', 'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
    'FG_PCT', 'FG3_PCT', 'FT_PCT', 'TS_PCT',
    'OFF_RATING', 'DEF_RATING', 'NET_RATING',
    'AST_PCT', 'REB_PCT', 'USG_PCT', 'PIE',
    'PTS_REB_AST']
for stat in z_stats:
    if stat in df.columns:
        df[f"{stat}_Z"] = df.groupby("SEASON")[stat].transform(
            lambda x: (x - x.mean())/x.std() if x.std() > 0 else 0
        )
print("✓ Engineered features")

# ------------------------
# Summary
# ------------------------
print("\n---- DATASET SUMMARY ----")
print(f"Total player-seasons: {len(df)}")
print(f"Players with MVP votes: {(df['MVP_SHARE'] > 0).sum()}")
print(f"MVP winners (>0.5 share): {(df['MVP_SHARE'] > 0.5).sum()}")

# ------------------------
# Export processed stats
# ------------------------
df.to_csv("nba_processed_stats.csv", index=False)
print(f"✓ Exported processed dataset")


In [ ]:
# ------------------------
# Define Features and Response Variable
# ------------------------
df = pd.read_csv("nba_processed_stats.csv")

features = [
    'GP', 'MIN', 'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV',
    'FG_PCT', 'FG3_PCT', 'FT_PCT', 'TS_PCT',
    'OFF_RATING', 'DEF_RATING', 'NET_RATING',
    'AST_PCT', 'REB_PCT', 'USG_PCT', 'PIE',
    'PTS_REB_AST', 'W_PCT',
    'PTS_Z', 'REB_Z', 'AST_Z', 'STL_Z', 'BLK_Z', 'NET_RATING_Z', 'PIE_Z', 'USG_PCT_Z', 
    'GP_Z', 'MIN_Z', 'TOV_Z',
    'FG_PCT_Z', 'FG3_PCT_Z', 'FT_PCT_Z', 'TS_PCT_Z',
    'OFF_RATING_Z', 'DEF_RATING_Z',
    'AST_PCT_Z', 'REB_PCT_Z',
    'PTS_REB_AST_Z'
]
X = df[features].fillna(df[features].median())
# Using MVP share as response to avoid having a binary variable of only 25 MVPs. This will allow the model
# to be more nuanced and robust.
y = df['MVP_SHARE']

In [ ]:
# ------------------------
# Split into Train and Test Data
# ------------------------

df['mvp_candidate'] = (df['MVP_SHARE'] > 0).astype(int)
# Stratified split to ensure that MVP candidates are 
# proportionally represented in each set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=df['mvp_candidate']  # ← Key parameter!
)

print(f"Train MVP candidates: {(y_train > 0).sum()} / {len(y_train)}")
print(f"Test MVP candidates: {(y_test > 0).sum()} / {len(y_test)}")

In [ ]:
# ------------------------
# Fit baseline linear regression
# ------------------------
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print("Baseline Linear Regression:")
print(f"R^2 score: {r2_score(y_test, y_pred):.3f}")
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.3f}")

In [ ]:
# ------------------------
# Fit random forest model
# ------------------------
rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Predictions
y_pred = rf_model.predict(X_test)

# Evaluation
print("Random Forest MVP Prediction:")
print(f"R^2 score: {r2_score(y_test, y_pred):.3f}")
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.3f}")

In [ ]:
# ------------------------
# Finding Features with > 1% importance
# ------------------------
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
threshold = 0.01
selected_features = importances[importances > threshold].index.tolist()
print(selected_features)

In [ ]:
# ------------------------
# Final Random Forest Model
# ------------------------
final_features = [
    'PTS_REB_AST_Z', 'W_PCT', 'NET_RATING_Z','GP_Z', 'AST_Z', 
    'PTS_Z', 'DEF_RATING_Z', 'FT_PCT_Z','PIE_Z', 'TS_PCT_Z'
]
# I decided to remove any non-standardized features to eliminate redundancy
X = df[final_features].fillna(df[final_features].median())
y = df['MVP_SHARE']

# Fresh split for final model
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df['mvp_candidate']
)

# Train final model
rf_final = RandomForestRegressor(
    n_estimators=500,
    max_depth=8,
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X_train, y_train)

# Predictions
y_pred = rf_final.predict(X_test)

# Evaluation
print("\nRandom Forest MVP Prediction (Reduced Features):")
print(f"R^2 score: {r2_score(y_test, y_pred):.3f}")
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.3f}")

# Feature importances
importances = pd.Series(rf_final.feature_importances_, index=final_features).sort_values(ascending=False)
print("\nTop Features:")
print(importances)

In [ ]:
# ------------------------
# Hyperparameter space
# ------------------------
param_grid = {
    'n_estimators': [200, 500, 800, 1000],
    'max_depth': [4, 6, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['auto', 'sqrt', 'log2']
}

# ------------------------
# Randomized Search CV
# ------------------------
rf = RandomForestRegressor(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_grid,
    n_iter=30,              # number of random combinations to try
    scoring='r2',           # evaluation metric
    cv=5,                   # 5-fold cross-validation
    verbose=2,
    random_state=42,
    n_jobs=-1
)

# Fit search
random_search.fit(X_train, y_train)

# Best parameters & score
print("Best Parameters:", random_search.best_params_)
print("Best CV R^2:", random_search.best_score_)

# Predict on test set with best model
best_rf = random_search.best_estimator_
y_pred_best = best_rf.predict(X_test)

rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
print(f"Test R^2: {r2_score(y_test, y_pred_best):.3f}")
print(f"Test RMSE: {rmse_best:.3f}")


In [ ]:
# ------------------------
# Final Tuned RF Model
# ------------------------
rf_model_tuned = random_search.best_estimator_
y_pred = rf_model_tuned.predict(X_test)

# ------------------------
# Predictions & evaluation
# ------------------------
print("Tuned Random Forest MVP Prediction:")
print(f"R^2 score: {r2_score(y_test, y_pred):.3f}")
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"RMSE: {rmse:.3f}")

# Feature importances
importances = pd.Series(rf_model_tuned.feature_importances_, index=final_features).sort_values(ascending=False)
print("\nTop Features:")
print(importances)

In [ ]:
# ------------------------
# Fetch 25-26 stats for final prediction
# ------------------------
df25_26 = collect_single_season(2025)

# Minimum games played to eliminate small samples
min_games = 10
df25_26 = df25_26[df25_26['GP'] >= min_games]

# ------------------------
# Engineer features and Z-scores to match model data
# ------------------------
df25_26['PTS_REB_AST'] = df25_26['PTS'] + df25_26['REB'] + df25_26['AST']

z_stats = ['PTS', 'REB', 'AST', 'PTS_REB_AST', 'GP', 'NET_RATING', 'DEF_RATING', 'FT_PCT', 'PIE', 'TS_PCT']
for stat in z_stats:
    mean_val = df25_26[stat].mean()
    std_val = df25_26[stat].std()
    df25_26[f'{stat}_Z'] = (df25_26[stat] - mean_val) / std_val if std_val > 0 else 0.0

X_25_26 = df25_26.reindex(columns=final_features, fill_value=0)
player_names = df25_26['PLAYER_NAME'].values


In [ ]:
# ------------------------
# Predict MVP share
# ------------------------
y_pred_25_26 = rf_model_tuned.predict(X_25_26)

# ------------------------
# Normalize across all players to get probability rather than share
# ------------------------
y_pred_norm = y_pred_25_26 / y_pred_25_26.sum()

# ------------------------
# Filter players with >=1% predicted probability for final output
# ------------------------
threshold = 0.01
top_mask = y_pred_norm >= threshold
top_player_names = [player_names[i] for i, flag in enumerate(top_mask) if flag]
top_probs = y_pred_norm[top_mask]

submission_25_26 = pd.DataFrame({
    'PLAYER_NAME': top_player_names,
    'MVP_PROBABILITY': top_probs
}).sort_values('MVP_PROBABILITY', ascending=False)

# ------------------------
# Save CSV and print top 10
# ------------------------
submission_25_26.to_csv("predictions.csv", index=False)
print(submission_25_26)


In [ ]:
# --------------------------------------
# Detailed evaluation on MVP candidates
# --------------------------------------

# Get the tuned model
rf_tuned = random_search.best_estimator_

# Predict on test set
y_pred = rf_tuned.predict(X_test)

print("\n" + "="*60)
print("MODEL PERFORMANCE BREAKDOWN")
print("="*60)

# Overall performance
r2_all = r2_score(y_test, y_pred)
rmse_all = np.sqrt(mean_squared_error(y_test, y_pred))
mae_all = mean_absolute_error(y_test, y_pred)

print(f"\n{'ALL PLAYERS:':<30}")
print(f"  Samples:     {len(y_test):,}")
print(f"  R² Score:    {r2_all:.3f}")
print(f"  RMSE:        {rmse_all:.3f}")
print(f"  MAE:         {mae_all:.3f}")

# MVP candidates only (any votes)
mvp_mask = y_test > 0
if mvp_mask.sum() > 0:
    y_test_mvp = y_test[mvp_mask]
    y_pred_mvp = y_pred[mvp_mask]
    
    r2_mvp = r2_score(y_test_mvp, y_pred_mvp)
    rmse_mvp = np.sqrt(mean_squared_error(y_test_mvp, y_pred_mvp))
    mae_mvp = mean_absolute_error(y_test_mvp, y_pred_mvp)
    
    print(f"\n{'MVP CANDIDATES (Share > 0):':<30}")
    print(f"  Samples:     {mvp_mask.sum()}")
    print(f"  R² Score:    {r2_mvp:.3f}")
    print(f"  RMSE:        {rmse_mvp:.3f}")
    print(f"  MAE:         {mae_mvp:.3f}")
    print(f"  Avg Share:   {y_test_mvp.mean():.3f}")
    print(f"  Max Share:   {y_test_mvp.max():.3f}")

# Strong candidates (>10% vote share)
strong_mask = y_test > 0.1
if strong_mask.sum() > 0:
    y_test_strong = y_test[strong_mask]
    y_pred_strong = y_pred[strong_mask]
    
    r2_strong = r2_score(y_test_strong, y_pred_strong)
    rmse_strong = np.sqrt(mean_squared_error(y_test_strong, y_pred_strong))
    mae_strong = mean_absolute_error(y_test_strong, y_pred_strong)
    
    print(f"\n{'STRONG CANDIDATES (Share > 0.1):':<30}")
    print(f"  Samples:     {strong_mask.sum()}")
    print(f"  R² Score:    {r2_strong:.3f}")
    print(f"  RMSE:        {rmse_strong:.3f}")
    print(f"  MAE:         {mae_strong:.3f}")
    print(f"  Avg Share:   {y_test_strong.mean():.3f}")

# Top contenders (>30% vote share)
contender_mask = y_test > 0.3
if contender_mask.sum() > 0:
    y_test_contender = y_test[contender_mask]
    y_pred_contender = y_pred[contender_mask]
    
    r2_contender = r2_score(y_test_contender, y_pred_contender)
    rmse_contender = np.sqrt(mean_squared_error(y_test_contender, y_pred_contender))
    mae_contender = mean_absolute_error(y_test_contender, y_pred_contender)
    
    print(f"\n{'TOP CONTENDERS (Share > 0.3):':<30}")
    print(f"  Samples:     {contender_mask.sum()}")
    print(f"  R² Score:    {r2_contender:.3f}")
    print(f"  RMSE:        {rmse_contender:.3f}")
    print(f"  MAE:         {mae_contender:.3f}")
    print(f"  Avg Share:   {y_test_contender.mean():.3f}")

# Winners (>50% vote share)
winner_mask = y_test > 0.5
if winner_mask.sum() > 0:
    print(f"\n{'LIKELY WINNERS (Share > 0.5):':<30}")
    print(f"  Samples:     {winner_mask.sum()}")
    print(f"  Avg Share:   {y_test[winner_mask].mean():.3f}")
    print(f"  Avg Pred:    {y_pred[winner_mask].mean():.3f}")

# -----------------------------------------
# Visual comparison
# -----------------------------------------
print("\n" + "="*60)
print("PERFORMANCE SUMMARY")
print("="*60)
print(f"{'Category':<30} {'Samples':<10} {'R²':<10} {'RMSE':<10}")
print("-" * 60)
print(f"{'All Players':<30} {len(y_test):<10} {r2_all:<10.3f} {rmse_all:<10.3f}")
if mvp_mask.sum() > 0:
    print(f"{'MVP Candidates (>0)':<30} {mvp_mask.sum():<10} {r2_mvp:<10.3f} {rmse_mvp:<10.3f}")
if strong_mask.sum() > 0:
    print(f"{'Strong Candidates (>0.1)':<30} {strong_mask.sum():<10} {r2_strong:<10.3f} {rmse_strong:<10.3f}")
if contender_mask.sum() > 0:
    print(f"{'Top Contenders (>0.3)':<30} {contender_mask.sum():<10} {r2_contender:<10.3f} {rmse_contender:<10.3f}")

In [ ]:
# ------------------------
# Citations
# ------------------------
# Sports Reference LLC. (2025). Basketball-Reference.com. https://www.basketball-reference.com
# National Basketball Association. (2025). NBA Stats API. https://stats.nba.com
# Portions of the code and analysis were assisted using ChatGPT - OpenAI (2025), 
# Claude — Anthropic (2025), and Copilot — Microsoft (2025)